# Supplementary Figure S3 — Full S12/S13 representation grid (fork AUC of OT flow J)

Sources: `benchmarks/synthetic_multimodal/s13/auc.csv` (4 embeddings × 4 whitening variants × 6 datasets) and
`benchmarks/synthetic_multimodal/s12/step0.csv` (scVI-10 and FA-20 with whitened cost, raw variant only).

In [1]:
from _style import *

prov = Provenance("FigS3", notebook="notebooks/supp_s13_whitening.ipynb", kind="supplementary")
a = pd.read_csv(prov.use("benchmarks/synthetic_multimodal/s13/auc.csv", "a", "S13"))
s12 = pd.read_csv(prov.use("benchmarks/synthetic_multimodal/s12/step0.csv", "a", "S12"))
V = prov.value
var = [("raw_s12", "raw"), ("whitened_full", "whitened embedding"), ("raw_ot_whitened_drift", "whitened drift fit only"),
       ("whitened_ot_raw_drift", "whitened OT cost only")]
embs = ["pca10", "pca20", "pca50", "fa20"]
dsets = sorted(a.dataset.unique())
rows, labels = [], []
for e in embs:
    for vv, vl in var:
        h = a[(a.embedding == e) & (a.variant == vv)].set_index("dataset").reindex(dsets).auc_lam1
        rows.append(h.values); labels.append(f"{e.upper()} · {vl}")
for e, lab in [("scvi10", "SCVI10 · raw (S12)"), ("fa20w", "FA20 · noise-whitened cost (S12)")]:
    h = s12[s12.embedding == e].assign(dataset=lambda x: x.preset + "_s" + x.seed.astype(str)).set_index("dataset").reindex(dsets).auc_lam1
    rows.append(h.values); labels.append(lab)
M = np.vstack(rows)
# check the raw S13 rows reproduce S12 step 0
for e in embs:
    r13 = a[(a.embedding == e) & (a.variant == "raw_s12")].set_index("dataset").auc_lam1
    r12 = s12[s12.embedding == e].assign(dataset=lambda x: x.preset + "_s" + x.seed.astype(str)).set_index("dataset").auc_lam1
    V(f"max_abs_diff_s13raw_vs_s12_{e}", float((r13 - r12.reindex(r13.index)).abs().max()))
fig, ax = plt.subplots(figsize=(4.6, 4.4))
cmap = matplotlib.colormaps["RdBu"]
im = ax.imshow(M, cmap=cmap, vmin=0, vmax=1, aspect="auto")
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        ax.text(j, i, f2(M[i, j]), ha="center", va="center", fontsize=5, color="white" if abs(M[i, j] - 0.5) > 0.32 else "black")
ax.set_xticks(range(len(dsets))); ax.set_xticklabels([d.replace("v3_", "") for d in dsets], rotation=40, ha="right")
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=5.5)
cb = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02); cb.set_label("fork AUC (0.5 = chance)")
prov.panel("a", "Fork AUC of OT flow J for every embedding x whitening variant x dataset (S13) plus scVI-10 and FA-20 whitened cost (S12)")
save(fig, "FigS3", kind="supplementary")
for lab, row in zip(labels, M):
    V(f"{lab}_min", float(np.nanmin(row))); V(f"{lab}_max", float(np.nanmax(row)))

[save] supplementary/FigS3: 139.5 x 101.2 mm, PNG 3295 x 2390 px @ 600 dpi


In [2]:
v = prov.values
prov.legend(f"""
**Supplementary Figure S3 | Representation grid for the OT flow operator (S12 Step 0, S13; pre-registered gates failed).** Fork AUC (λ₁, saddle vs committed)
of OT flow J on v3_default and v3_transition seeds 0–2 for PCA-10/20/50 and FA-20, each with four variants: raw coordinates; whitened embedding (OT and
drift both in whitened coordinates); whitened drift fit with raw-coordinate OT; whitened OT cost with raw-coordinate drift. Bottom rows: scVI-10 and FA-20 with
an OT cost whitened by the FA noise variances (S12 item d). The S13 raw rows reproduce S12 (max |ΔAUC| over PCA-10/20/50/FA-20:
{f2(max(v[f'max_abs_diff_s13raw_vs_s12_{e}'] for e in ['pca10', 'pca20', 'pca50', 'fa20']), 4)}). Colour: blue > 0.5 (saddle cells ranked above committed), red < 0.5 (inverted).
""")
prov.write()